# FedShield · BAF (onboarding fraud, plug-in adapter demo)

The same federation core on a dataset with no transactions at all: account-opening applications, three banks by payment type. Demonstrates the adapter contract (`fedshield/adapters.py`).

Attach: the BAF dataset and the FedShield code dataset.

In [ ]:
# --- setup: put the FedShield code on the path and point it at the Kaggle inputs ---
import os, sys, glob, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb>=1.1", "numba>=0.60", "xgboost>=2.0"], check=True)
# the repo attached as a Kaggle dataset (any slug): find the folder that contains fedshield/
code = next(p for p in glob.glob("/kaggle/input/*/") + glob.glob("/kaggle/input/*/*/") if os.path.isdir(os.path.join(p, "fedshield")))
sys.path.insert(0, code)
os.environ["FEDSHIELD_WORK"] = "/kaggle/working/work"
os.environ["FEDSHIELD_RESULTS"] = "/kaggle/working/results"
os.environ["FEDSHIELD_DUCKDB_MEM"] = "12GB"        # Kaggle has ~30 GB RAM
print("code:", code)


In [ ]:
os.environ["FEDSHIELD_BAF_DIR"] = next(p for p in glob.glob("/kaggle/input/*/") if os.path.exists(os.path.join(p, "Base.csv")))

In [ ]:
!cd {code} && python -m fedshield.ingest.baf Base

In [ ]:
from fedshield.api.pipeline import LiveRun
run = LiveRun(); run.start(dataset='baf', sample=200000, n_trees=60, pace=0, attack='gradient_scale'); run.thread.join()
trees = [e for e in run.events if e['stage'] == 'tree' and 'eval_auprc' in e]
setup = next(e for e in run.events if e['stage'] == 'setup')
print('gate 2:', {b: g['rejected'] for b, g in setup['gate2'].items()})
print('final AUPRC', trees[-1]['eval_auprc'])